
# Week 2 - Regularized Regression

## Introduction

Lasso, Ridge, and Elastic Net regression to three digital marketing datasets.

### Datasets
1. Predict Conversion in Digital Marketing Dataset
2. Marketing and Product Performance
3. Companies (Synthetic B2B CRM & Marketing Dataset)


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error

In [2]:
df1 = pd.read_csv("digital_marketing_campaign_dataset.csv")
df2 = pd.read_csv("marketing_and_product_performance.csv")
df3 = pd.read_csv("companies_clean_734.csv") # Synthetic B2B CRM & Marketing Dataset

print("Digital Marketing Campaign:", df1.shape)
print("Marketing and Product Performance:", df2.shape)
print("Synthetic B2B CRM & Marketing Dataset:", df3.shape)

Digital Marketing Campaign: (8000, 20)
Marketing and Product Performance: (10000, 17)
Synthetic B2B CRM & Marketing Dataset: (734, 19)


## Dataset 1: Digital Marketing Campaign

In [3]:
# selecting predictors and target
features1 = [
    "AdSpend",
    "ClickThroughRate",
    "WebsiteVisits",
    "TimeOnSite",
    "EmailClicks"
]

X1 = df1[features1]
y1 = df1["ConversionRate"]

#splitting the data
X1_train, X1_test, y1_train, y1_test = train_test_split(
    X1,
    y1,
    test_size=0.2,
    random_state=42
)

# baseline linear regression
baseline1 = LinearRegression()
baseline1.fit(X1_train, y1_train)
baseline1_pred = baseline1.predict(X1_test)

print("Training R²:", round(baseline1.score(X1_train, y1_train), 4))
print("Testing R²:", round(r2_score(y1_test, baseline1_pred), 4))
print("Testing RMSE:", round(np.sqrt(mean_squared_error(y1_test, baseline1_pred)), 4))

Training R²: 0.0007
Testing R²: -0.0009
Testing RMSE: 0.0541


### Lasso Regression

In [4]:
# lasso pipeline
lasso_pipe1 = Pipeline([
    ("scaler", StandardScaler()),
    ("lasso", Lasso(max_iter=10000))
])

# several alpha values
lasso_params1 = {
    "lasso__alpha": [0.0001, 0.001, 0.01, 0.1, 1, 10]
}

lasso_grid1 = GridSearchCV(
    lasso_pipe1,
    lasso_params1,
    cv=5,
    scoring="neg_mean_squared_error"
)
lasso_grid1.fit(X1_train, y1_train)
#best model
lasso_pred1 = lasso_grid1.predict(X1_test)

print("Best alpha:", lasso_grid1.best_params_["lasso__alpha"])
print("Training R²:", round(r2_score(y1_train, lasso_grid1.predict(X1_train)), 4))
print("Testing R²:", round(r2_score(y1_test, lasso_pred1), 4))
print("Testing RMSE:", round(np.sqrt(mean_squared_error(y1_test, lasso_pred1)), 4))

Best alpha: 0.001
Training R²: 0.0001
Testing R²: -0.0017
Testing RMSE: 0.0541


In [5]:
#Lasso coefficients
lasso_coefficients1 = pd.DataFrame({"Feature": features1, "Coefficient": lasso_grid1.best_estimator_ .named_steps["lasso"].coef_})

print(lasso_coefficients1.round(6).to_string(index=False))
print("\nFeatures retained:", (lasso_coefficients1["Coefficient"] != 0).sum())
print("Features eliminated:", (lasso_coefficients1["Coefficient"] == 0).sum())

         Feature  Coefficient
         AdSpend    -0.000176
ClickThroughRate    -0.000000
   WebsiteVisits    -0.000000
      TimeOnSite     0.000000
     EmailClicks     0.000000

Features retained: 1
Features eliminated: 4


### Lasso Regression Findings

Cross-validation selected an optimal alpha of 0.001. The model achieved a training R² of 0.0001, a testing R² of -0.0017, and a testing RMSE of 0.0541.
Lasso eliminated four of the five predictors, retaining only AdSpend. This demonstrates how Lasso can perform automatic feature selection by reducing less useful coefficients to zero.
Lasso did not improve predictive performance compared to the baseline linear regression model.


### Ridge Regression

In [6]:
#creating Ridge pipeline
ridge_pipe1 = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge())
])

#alpha values
ridge_params1 = {
    "ridge__alpha": [0.0001, 0.001, 0.01, 0.1, 1, 10, 100, 1000]
}

#finding the best alpha using cross-validation
ridge_grid1 = GridSearchCV(
    ridge_pipe1,
    ridge_params1,
    cv=5,
    scoring="neg_mean_squared_error"
)

ridge_grid1.fit(X1_train, y1_train)

# evaluating
ridge_pred1 = ridge_grid1.predict(X1_test)

print("Best alpha:", ridge_grid1.best_params_["ridge__alpha"])
print("Training R²:", round(r2_score(y1_train, ridge_grid1.predict(X1_train)), 4))
print("Testing R²:", round(r2_score(y1_test, ridge_pred1), 4))
print("Testing RMSE:", round(np.sqrt(mean_squared_error(y1_test, ridge_pred1)), 4))

Best alpha: 1000
Training R²: 0.0007
Testing R²: -0.001
Testing RMSE: 0.0541


In [7]:
#Ridge coefficients
ridge_coefficients1 = pd.DataFrame({
    "Feature": features1,
    "Coefficient": ridge_grid1.best_estimator_
        .named_steps["ridge"].coef_
})

print(ridge_coefficients1.round(6).to_string(index=False))
print("\nFeatures retained:", (ridge_coefficients1["Coefficient"] != 0).sum())

         Feature  Coefficient
         AdSpend    -0.001018
ClickThroughRate    -0.000247
   WebsiteVisits    -0.000361
      TimeOnSite     0.000296
     EmailClicks     0.000435

Features retained: 5



### Ridge Regression Findings

Cross-validation selected an alpha of 1000. Ridge achieved a training R² of 0.0007, a testing R² of -0.0010, and a testing RMSE of 0.0541.
Unlike Lasso, Ridge retained all five predictors while shrinking their coefficients. This demonstrates how Ridge reduces the influence of predictors without eliminating them. Ridge performed slightly better than Lasso but produced results nearly identical to the baseline linear regression model. Overall, regularization has not meaningfully improved predictive performance.


### Elastic Net Regression

In [8]:
#creating Elastic Net pipeline
elastic_pipe1 = Pipeline([
    ("scaler", StandardScaler()),
    ("elastic", ElasticNet(max_iter=10000))
])

#define parameters
elastic_params1 = {
    "elastic__alpha": [0.0001, 0.001, 0.01, 0.1, 1, 10],
    "elastic__l1_ratio": [0.1, 0.3, 0.5, 0.7, 0.9]
}

#finding the best parameters
elastic_grid1 = GridSearchCV(
    elastic_pipe1,
    elastic_params1,
    cv=5,
    scoring="neg_mean_squared_error"
)

elastic_grid1.fit(X1_train, y1_train)

#evaluating model
elastic_pred1 = elastic_grid1.predict(X1_test)

print("Best alpha:", elastic_grid1.best_params_["elastic__alpha"])
print("Best l1_ratio:", elastic_grid1.best_params_["elastic__l1_ratio"])
print("Training R²:", round(r2_score(y1_train, elastic_grid1.predict(X1_train)), 4))
print("Testing R²:", round(r2_score(y1_test, elastic_pred1), 4))
print("Testing RMSE:", round(np.sqrt(mean_squared_error(y1_test, elastic_pred1)), 4))

Best alpha: 0.001
Best l1_ratio: 0.7
Training R²: 0.0003
Testing R²: -0.0016
Testing RMSE: 0.0541


In [9]:
#examine Elastic Net coefficients
elastic_coefficients1 = pd.DataFrame({
    "Feature": features1,
    "Coefficient": elastic_grid1.best_estimator_
        .named_steps["elastic"].coef_
})

print(elastic_coefficients1.round(6).to_string(index=False))
print("\nFeatures retained:", (elastic_coefficients1["Coefficient"] != 0).sum())
print("Features eliminated:",(elastic_coefficients1["Coefficient"] == 0).sum())

         Feature  Coefficient
         AdSpend    -0.000476
ClickThroughRate    -0.000000
   WebsiteVisits    -0.000000
      TimeOnSite     0.000000
     EmailClicks     0.000000

Features retained: 1
Features eliminated: 4


### Elastic Net Regression Findings

Cross-validation selected an alpha of 0.001 and an l1 ratio of 0.7. The model achieved a training R² of 0.0003, a testing R² of -0.0016, and a testing RMSE of 0.0541.
Elastic Net retained only AdSpend and reduced the other four predictor coefficients to zero. This result was similar to Lasso, which also retained only one predictor.
Overall, Elastic Net did not improve predictive performance compared with the baseline linear regression model.

### Digital Marketing Campaign Dataset: Conclusion

Regularized regression did not meaningfully improve prediction of ConversionRate.

Lasso and Elastic Net both performed feature selection by retaining only AdSpend, while Ridge retained all five predictors and reduced their coefficients.

The baseline linear regression model produced the highest testing R², although all models had very low predictive performance. These results suggest that regularization does not solve the weak predictive relationship between the selected features and ConversionRate.

## Dataset 2: Marketing and Product Performance

This analysis applies Lasso, Ridge, and Elastic Net regression to the Marketing and Product Performance dataset. The target variable is Revenue_Generated. The predictors are Budget, Clicks, Conversions, Units_Sold, and Bundle_Price.

In [10]:
#selecting predictors and target
features2 = [
    "Budget",
    "Clicks",
    "Conversions",
    "Units_Sold",
    "Bundle_Price"
]

X2 = df2[features2]
y2 = df2["Revenue_Generated"]

# spliting data
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2,
    y2,
    test_size=0.2,
    random_state=42
)

# baseline linear regression
baseline2 = LinearRegression()
baseline2.fit(X2_train, y2_train)

baseline2_pred = baseline2.predict(X2_test)

print("Training R²:", round(baseline2.score(X2_train, y2_train), 4))
print("Testing R²:", round(r2_score(y2_test, baseline2_pred), 4))
print("Testing RMSE:", round(np.sqrt(mean_squared_error(y2_test, baseline2_pred)), 4))

Training R²: 0.0007
Testing R²: -0.0012
Testing RMSE: 28572.378


### Lasso Regression

In [11]:
# creating Lasso pipeline
lasso_pipe2 = Pipeline([
    ("scaler", StandardScaler()),
    ("lasso", Lasso(max_iter=10000))
])
# alpha values
lasso_params2 = {
    "lasso__alpha": [0.01, 0.1, 1, 10, 100, 1000, 10000]
}

# finding best alpha cv
lasso_grid2 = GridSearchCV(
    lasso_pipe2,
    lasso_params2,
    cv=5,
    scoring="neg_mean_squared_error"
)

lasso_grid2.fit(X2_train, y2_train)

# evaluating
lasso_pred2 = lasso_grid2.predict(X2_test)

print("Best alpha:",lasso_grid2.best_params_["lasso__alpha"])
print("Training R²:", round(r2_score(y2_train, lasso_grid2.predict(X2_train)), 4))
print("Testing R²:", round(r2_score(y2_test, lasso_pred2), 4))
print("Testing RMSE:", round(np.sqrt(mean_squared_error(y2_test, lasso_pred2)), 4))

Best alpha: 10000
Training R²: 0.0
Testing R²: -0.0008
Testing RMSE: 28567.7879


In [12]:
# Lasso coefficients
lasso_coefficients2 = pd.DataFrame({
    "Feature": features2,
    "Coefficient": lasso_grid2.best_estimator_
        .named_steps["lasso"].coef_
})

print(lasso_coefficients2.round(6).to_string(index=False))
print("\nFeatures retained:", (lasso_coefficients2["Coefficient"] != 0).sum())
print("Features eliminated:", (lasso_coefficients2["Coefficient"] == 0).sum())


     Feature  Coefficient
      Budget         -0.0
      Clicks         -0.0
 Conversions         -0.0
  Units_Sold          0.0
Bundle_Price          0.0

Features retained: 0
Features eliminated: 5



### Lasso Regression Findings

Cross-validation selected an alpha of 10000. The model achieved a training R² of 0.0000, a testing R² of -0.0008, and a testing RMSE of 28567.7879. Lasso eliminated all five predictors, demonstrating the effects of strong regularization. The resulting model effectively predicts a constant value rather than using the selected features. Although Lasso slightly improved testing performance compared to the baseline, neither model demonstrated meaningful predictive ability.


In [13]:
# Ridge pipeline
ridge_pipe2 = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge())
])

# alpha values
ridge_params2 = {
    "ridge__alpha": [
        0.01, 0.1, 1, 10, 100,
        1000, 10000, 100000
    ]
}

# cv
ridge_grid2 = GridSearchCV(
    ridge_pipe2,
    ridge_params2,
    cv=5,
    scoring="neg_mean_squared_error"
)

ridge_grid2.fit(X2_train, y2_train)

# evaluate
ridge_pred2 = ridge_grid2.predict(X2_test)

print("Best alpha:", ridge_grid2.best_params_["ridge__alpha"])
print("Training R²:", round( r2_score(y2_train, ridge_grid2.predict(X2_train)), 4))
print("Testing R²:", round(r2_score(y2_test, ridge_pred2), 4))
print("Testing RMSE:", round(np.sqrt(mean_squared_error(y2_test, ridge_pred2)), 4))

Best alpha: 100000
Training R²: 0.0001
Testing R²: -0.0008
Testing RMSE: 28567.4615


In [14]:
#Ridge coefficients
ridge_coefficients2 = pd.DataFrame({
    "Feature": features2,
    "Coefficient": ridge_grid2.best_estimator_
        .named_steps["ridge"].coef_
})

print(ridge_coefficients2.round(6).to_string(index=False))
print("\nFeatures retained:", (ridge_coefficients2["Coefficient"] != 0).sum())

     Feature  Coefficient
      Budget   -55.100797
      Clicks    -8.310792
 Conversions    -5.569942
  Units_Sold     6.341224
Bundle_Price     1.386082

Features retained: 5


### Ridge Regression Findings

Cross-validation selected an alpha of 100000. The model achieved a training R² of 0.0001, a testing R² of -0.0008, and a testing RMSE of 28567.4615. Ridge retained all five predictors while shrinking their coefficients. This differs from Lasso, which eliminated all predictors. Although Ridge produced the lowest testing RMSE so far for this dataset, the improvement was extremely small and the negative testing R² indicates that predictive performance remains weak.

### Elastic Net Regression

In [15]:
# Elastic Net pipeline
elastic_pipe2 = Pipeline([
    ("scaler", StandardScaler()),
    ("elastic", ElasticNet(max_iter=10000))
])

# parameter grid
elastic_params2 = {
    "elastic__alpha": [
        0.01, 0.1, 1, 10, 100,
        1000, 10000
    ],
    "elastic__l1_ratio": [
        0.1, 0.3, 0.5, 0.7, 0.9
    ]
}

# best parameters
elastic_grid2 = GridSearchCV( elastic_pipe2, elastic_params2, cv=5, scoring="neg_mean_squared_error")
elastic_grid2.fit(X2_train, y2_train)

#model
elastic_pred2 = elastic_grid2.predict(X2_test)

print("Best alpha:", elastic_grid2.best_params_["elastic__alpha"])
print("Best l1_ratio:", elastic_grid2.best_params_["elastic__l1_ratio"])
print("Training R²:", round(r2_score(y2_train, elastic_grid2.predict(X2_train)), 4))
print("Testing R²:", round(r2_score(y2_test, elastic_pred2), 4))
print("Testing RMSE:", round(np.sqrt(mean_squared_error(y2_test, elastic_pred2)), 4))

Best alpha: 1000
Best l1_ratio: 0.3
Training R²: 0.0
Testing R²: -0.0008
Testing RMSE: 28567.786


In [16]:
#examine Elastic Net coefficients
elastic_coefficients2 = pd.DataFrame({
    "Feature": features2,
    "Coefficient": elastic_grid2.best_estimator_
        .named_steps["elastic"].coef_
})

print(elastic_coefficients2.round(6).to_string(index=False))
print("\nFeatures retained:", (elastic_coefficients2["Coefficient"] != 0).sum())
print("Features eliminated:", (elastic_coefficients2["Coefficient"] == 0).sum())

     Feature  Coefficient
      Budget    -0.632737
      Clicks    -0.000000
 Conversions    -0.000000
  Units_Sold     0.000000
Bundle_Price     0.000000

Features retained: 1
Features eliminated: 4



### Elastic Net Regression Findings

Cross-validation selected an alpha of 1000 and an l1 ratio of 0.3. The model achieved a training R² of 0.0000, a testing R² of -0.0008, and a testing RMSE of 28567.7860. Elastic Net retained only Budget and eliminated the other four predictors. Unlike Lasso, which eliminated all five predictors, Elastic Net retained one. Ridge retained all five. Although Elastic Net slightly improved testing performance compared to the baseline, its negative testing R² indicates that predictive performance remains weak.



### Marketing and Product Performance Dataset: Conclusion

Regularized regression produced only minor improvements in predicting Revenue_Generated.

Lasso eliminated all five predictors, Ridge retained all five, and Elastic Net retained only Budget. These results demonstrate the different approaches each method takes toward coefficient shrinkage and feature selection.

Ridge achieved the lowest testing RMSE of 28567.4615, although its improvement over the other models was negligible.

Overall, regularization did not meaningfully improve predictive performance. The selected predictors appear to provide limited information for predicting revenue.



## Dataset 3: Synthetic B2B CRM & Marketing Dataset

In [17]:
# select predictors and target
features3 = [
    "Annual_Revenue (M₺)",
    "Marketing_Spend (K₺)",
    "Days_Since_Last_Purchase",
    "Total_Purchases_Last_Year"
]

X3 = df3[features3]
y3 = df3["Leads_Generated"]

#splitting data
X3_train, X3_test, y3_train, y3_test = train_test_split(X3, y3, test_size=0.2, random_state=42)

# baseline linear regression
baseline3 = LinearRegression()
baseline3.fit(X3_train, y3_train)
baseline3_pred = baseline3.predict(X3_test)

print("Training R²:", round(baseline3.score(X3_train, y3_train), 4))
print("Testing R²:", round(r2_score(y3_test, baseline3_pred), 4))
print("Testing RMSE:", round(np.sqrt(mean_squared_error(y3_test, baseline3_pred)), 4))

Training R²: 0.0232
Testing R²: -0.0218
Testing RMSE: 3.0941


### Lasso Regression

In [18]:
# Lasso pipeline
lasso_pipe3 = Pipeline([
    ("scaler", StandardScaler()),
    ("lasso", Lasso(max_iter=10000))
])

lasso_params3 = {
    "lasso__alpha": [
        0.0001, 0.001, 0.01,
        0.1, 1, 10, 100
    ]
}

#cv
lasso_grid3 = GridSearchCV(lasso_pipe3, lasso_params3, cv=5, scoring="neg_mean_squared_error")
lasso_grid3.fit(X3_train, y3_train)

lasso_pred3 = lasso_grid3.predict(X3_test)
print("Best alpha:", lasso_grid3.best_params_["lasso__alpha"])
print("Training R²:", round(r2_score(y3_train, lasso_grid3.predict(X3_train)), 4))
print("Testing R²:", round(r2_score(y3_test, lasso_pred3), 4))
print("Testing RMSE:", round(np.sqrt(mean_squared_error(y3_test, lasso_pred3)), 4))

Best alpha: 0.1
Training R²: 0.0209
Testing R²: -0.0043
Testing RMSE: 3.0675


In [19]:
# Lasso coefficients
lasso_coefficients3 = pd.DataFrame({
    "Feature": features3,
    "Coefficient": lasso_grid3.best_estimator_
        .named_steps["lasso"].coef_
})

print(lasso_coefficients3.round(6).to_string(index=False))
print("\nFeatures retained:",(lasso_coefficients3["Coefficient"] != 0).sum())
print("Features eliminated:",(lasso_coefficients3["Coefficient"] == 0).sum())

                  Feature  Coefficient
      Annual_Revenue (M₺)     0.158835
     Marketing_Spend (K₺)     0.247140
 Days_Since_Last_Purchase     0.000000
Total_Purchases_Last_Year    -0.016803

Features retained: 3
Features eliminated: 1



### Lasso Regression Findings

Cross-validation selected an alpha of 0.1. The model achieved a training R² of 0.0209, a testing R² of -0.0043, and a testing RMSE of 3.0675.Lasso retained three predictors: Annual Revenue, Marketing Spend, and Total Purchases Last Year. It eliminated Days Since Last Purchase by reducing its coefficient to zero. Compared to the baseline linear regression model, Lasso improved testing performance. However, the negative testing R² indicates that the model still has limited predictive ability.


In [20]:
# Ridge pipeline
ridge_pipe3 = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge())
])

# alpha
ridge_params3 = {"ridge__alpha": [0.0001, 0.001, 0.01, 0.1, 1, 10, 100, 1000]}
#best alpha
ridge_grid3 = GridSearchCV(ridge_pipe3, ridge_params3, cv=5,scoring="neg_mean_squared_error")
ridge_grid3.fit(X3_train, y3_train)

#eval
ridge_pred3 = ridge_grid3.predict(X3_test)

print("Best alpha:",ridge_grid3.best_params_["ridge__alpha"])
print("Training R²:", round(r2_score(y3_train, ridge_grid3.predict(X3_train)), 4))
print("Testing R²:", round(r2_score(y3_test, ridge_pred3), 4))
print("Testing RMSE:", round(np.sqrt(mean_squared_error(y3_test, ridge_pred3)), 4))

Best alpha: 1000
Training R²: 0.0167
Testing R²: -0.0084
Testing RMSE: 3.0738


In [21]:
# Ridge coefficients
ridge_coefficients3 = pd.DataFrame({
    "Feature": features3,
    "Coefficient": ridge_grid3.best_estimator_
        .named_steps["ridge"].coef_
})

print(ridge_coefficients3.round(6).to_string(index=False))
print("\nFeatures retained:", (ridge_coefficients3["Coefficient"] != 0).sum())

                  Feature  Coefficient
      Annual_Revenue (M₺)     0.117477
     Marketing_Spend (K₺)     0.136800
 Days_Since_Last_Purchase     0.029519
Total_Purchases_Last_Year    -0.033394

Features retained: 4


### Ridge Regression Findings

Cross-validation selected an alpha of 1000. The model achieved a training R² of 0.0167, a testing R² of -0.0084, and a testing RMSE of 3.0738. Ridge retained all four predictors while shrinking their coefficients. This differs from Lasso, which eliminated Days Since Last Purchase. Ridge improved testing performance compared with the baseline linear regression model, although Lasso still produced slightly better testing results. The negative testing R² indicates that predictive performance remains limited.

### Elastic Net Regression

In [22]:
# Elastic Net Regression
elastic_pipe3 = Pipeline([
    ("scaler", StandardScaler()),
    ("elastic", ElasticNet(max_iter=10000))
])

elastic_params3 = {"elastic__alpha": [0.0001, 0.001, 0.01, 0.1, 1, 10, 100], "elastic__l1_ratio": [0.1, 0.3, 0.5, 0.7, 0.9]}

elastic_grid3 = GridSearchCV(
    elastic_pipe3, elastic_params3, cv=5,
    scoring="neg_mean_squared_error"
)

elastic_grid3.fit(X3_train, y3_train)
elastic_pred3 = elastic_grid3.predict(X3_test)

print("Best alpha:", elastic_grid3.best_params_["elastic__alpha"])
print("Best l1_ratio:", elastic_grid3.best_params_["elastic__l1_ratio"])
print("Training R²:", round(r2_score(y3_train, elastic_grid3.predict(X3_train)), 4))
print("Testing R²:", round(r2_score(y3_test, elastic_pred3), 4))
print("Testing RMSE:", round(np.sqrt(mean_squared_error(y3_test, elastic_pred3)), 4))

Best alpha: 1
Best l1_ratio: 0.1
Training R²: 0.016
Testing R²: -0.0018
Testing RMSE: 3.0637


In [23]:
# Elastic Net coefficients
elastic_coefficients3 = pd.DataFrame({
    "Feature": features3,
    "Coefficient": elastic_grid3.best_estimator_.named_steps["elastic"].coef_
})

print(elastic_coefficients3.round(6).to_string(index=False))
print("\nFeatures retained:", (elastic_coefficients3["Coefficient"] != 0).sum())
print("Features eliminated:", (elastic_coefficients3["Coefficient"] == 0).sum())


                  Feature  Coefficient
      Annual_Revenue (M₺)     0.112895
     Marketing_Spend (K₺)     0.142826
 Days_Since_Last_Purchase     0.000103
Total_Purchases_Last_Year    -0.006799

Features retained: 4
Features eliminated: 0


### Elastic Net Regression Findings

Cross-validation selected an alpha of 1 and an l1 ratio of 0.1. The model achieved a training R² of 0.0160, a testing R² of -0.0018, and a testing RMSE of 3.0637. Elastic Net retained all four predictors, although Days Since Last Purchase had a coefficient close to zero. This differs from Lasso, which eliminated that predictor entirely. Elastic Net produced the lowest testing RMSE among the four models. However, its negative testing R² indicates that overall predictive performance remains limited.

### Synthetic B2B CRM & Marketing Dataset: Conclusion

Regularization improved testing performance when predicting Leads_Generated, although all models produced negative testing R² values.

Lasso retained three predictors, eliminating Days Since Last Purchase. Ridge and Elastic Net retained all four predictors while shrinking their coefficients.

Elastic Net achieved the lowest testing RMSE of 3.0637, followed by Lasso at 3.0675 and Ridge at 3.0738.

These results suggest that regularization helped reduce the effects of model complexity, but the selected predictors still provided limited predictive information.



## Week 2: Overall Findings

The analysis of this week was done for the three datasets on digital marketing using Lasso, Ridge, and Elastic Net regression. In order to obtain the regularization parameters, cross-validation was used, and training R², testing R², and RMSE were used as the measures of model performance. It was observed that different regularization methods were used for the selection of features in the models. While Lasso reduced the number of predictors in many cases, Ridge kept all the predictors. Elastic Net had features from both Lasso and Ridge. The improvements that came from regularization were minimal in the first two datasets. In the Companies dataset, there were some improvements made, and Elastic Net showed the least testing RMSE. Yet, the testing R² was negative for all the models, which indicated poor predictive performance.
